In [2]:
!pip install korpora konlpy gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 44.4 MB/s eta 0:00:00


In [1]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt
from collections import Counter
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from numpy.linalg import norm
import gensim
from gensim.models import Word2Vec

# =====================================================================
# [1] 임베딩 클래스
# =====================================================================
num_embeddings = 5000
embedding_dim = 128
embedding = torch.nn.Embedding(
    num_embeddings,
    embedding_dim,
    padding_idx=None,
    max_norm=None,
    norm_type=2.0
)

# =====================================================================
# [2] 예제 6.3 기본 Skip-gram 클래스
# =====================================================================
class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )
        self.linear = nn.Linear(
            in_features=embedding_dim,
            out_features=vocab_size
        )

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

# =====================================================================
# [3] 예제 6.4 영화 리뷰 데이터셋 전처리
# =====================================================================
corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

# =====================================================================
# [4] 예제 6.5 단어 사전 구축
# =====================================================================
def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens_list in corpus:
        counter.update(tokens_list)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

# =====================================================================
# [5] 예제 6.6 Skip-gram의 단어 쌍 추출
# =====================================================================
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs

word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

# =====================================================================
# [6] 예제 6.7 인덱스 쌍 변환
# =====================================================================
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])

# =====================================================================
# [7] 데이터로더 적용
# =====================================================================
index_pairs = torch.tensor(index_pairs)
center_indexs = index_pairs[:, 0]
contenxt_indexs = index_pairs[:, 1]

dataset = TensorDataset(center_indexs, contenxt_indexs)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# =====================================================================
# [8] Skip-gram 모델 준비 작업
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

# =====================================================================
# [9] 모델 학습
# =====================================================================
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(dataloader)
    print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

# =====================================================================
# [10] 임베딩 값 추출
# =====================================================================
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding_val in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding_val

index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

# =====================================================================
# [11] 단어 임베딩 유사도 계산
# =====================================================================
def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1 : n + 1]
    return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

# =====================================================================
# [12] Gensim Word2Vec 클래스 구조 예시
# =====================================================================
word2vec = gensim.models.Word2Vec(
    sentences=None,
    corpus_file=None,
    vector_size=100,
    alpha=0.025,
    window=5,
    min_count=5,
    workers=3,
    sg=0,
    hs=0,
    cbow_mean=1,
    negative=5,
    ns_exponent=0.75,
    max_final_vocab=None,
    epochs=5,
    batch_words=10000
)

# =====================================================================
# [13] 예제 6.13 Word2Vec 모델 학습
# =====================================================================
word2vec = Word2Vec(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    epochs=3,
    max_final_vocab=10000
)

# word2vec.save("../models/word2vec.model")
# word2vec = Word2Vec.load("../models/word2vec.model")

# =====================================================================
# [14] 단어 임베딩 추출 및 유사도 계산 예제
# =====================================================================
word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2="연기력"))


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 88.4MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 34.2MB/s]                            


[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]
['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001
[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]
[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
Epoch :    1, Cost : 6.196
Epoch :    2, Cost : 5.980
Epoch :    3, Cost : 5.930
Epoch :    4, Cost : 5.900
Epoch :    5, Cost : 5.878
Epoch :    6, Cost : 5.861
Epoch :    7, Cost : 5.846
Epoch :    8, Cost : 5.833
Epoch :    9, Cost : 5.822
Epoch :   10, Cost : 5.811
연기
[-0.62757164  0.47970012 -0.54531    -0.58930236 -0.9557295   0.08021533
 -0.52800137  1.7617725   0.6310371   2.0121963   0.34239867 -0.39904076
 -0.3492029   0.9759213   0.98989654 -0.56674516 -0.5156166   0.89194393
  0.7170068   1.155815   -0.41376138 -1.7628882   0.5046302  -0.66317713
 -0.50419676 -0.19724233  0.43929923 -1.8638272  -0.8608403   1.3380901
  0.32825705  0.3189446  -1.3768436  -0.623